In [1]:
!pip install mlflow

In [37]:
import mlflow
import mlflow.sklearn
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split
import seaborn as sns
import matplotlib.pyplot as plt
import mlflow.xgboost
from skopt import BayesSearchCV
from skopt.space import Real, Integer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
from sklearn.feature_extraction.text import CountVectorizer

In [2]:
data = pd.read_csv("/kaggle/input/datasets/bjdhdhjdbd/twitter-data-sentiment/clean_data.csv")
data.head()

,clean_text,category
0,family mormon never try explain still stare pu...,1.0
1,buddhism much lot compatible christianity espe...,1.0
2,seriously say thing first get complex explain ...,-1.0
3,learn want teach different focus goal not wrap...,0.0
4,benefit may want read live buddha live christ ...,1.0


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199702 entries, 0 to 199701
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   clean_text  199508 non-null  object 
 1   category    199702 non-null  float64
dtypes: float64(1), object(1)
memory usage: 3.0+ MB


In [3]:
data["clean_text"].isna().sum()
data = data.dropna()

In [4]:
data["clean_text"].isna().sum()

np.int64(0)

In [5]:
vectorizer = CountVectorizer(max_features=1000)

In [6]:

X = vectorizer.fit_transform(data['clean_text']).toarray()

In [7]:
X.shape

(199508, 1000)

In [8]:
y = data["category"]
y.shape

(199508,)

In [9]:
mlflow.set_tracking_uri("http://ec2-13-50-105-122.eu-north-1.compute.amazonaws.com:5000/")
mlflow.set_experiment("XGBoost base_line ")

<Experiment: artifact_location='s3://zg-mlflow/1', creation_time=1785005170371, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1785005170371, lifecycle_stage='active', name='XGBoost base_line ', tags={}, trace_location=None, workspace='default'>

In [10]:
!pip install boto3

In [13]:
!pip install awscli

In [12]:
x_train , x_test, y_train  , y_test = train_test_split(X , y , test_size=0.2, random_state=42)
x_train.shape
y_train.shape

(159606,)

In [17]:
mapping = {
    -1: 0,
     0: 1,
     1: 2
}

y_train = y_train.map(mapping)
y_test = y_test.map(mapping)

In [13]:
xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42
)

In [14]:
search_space = {

    "max_depth": Integer(3, 12),

    "learning_rate": Real(0.01, 0.3),

    "n_estimators": Integer(100, 800),

    "subsample": Real(0.5, 1.0),

    "colsample_bytree": Real(0.5, 1.0),

    "gamma": Real(0, 5),

    "min_child_weight": Integer(1, 10)
}

In [19]:
bayes_search = BayesSearchCV(

    estimator=xgb,

    search_spaces=search_space,

    n_iter=5,

    cv=5,

    scoring="accuracy",

    n_jobs=-1,

    random_state=42,

    verbose=2
)

In [20]:
bayes_search.fit(x_train, y_train)

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
[CV] END colsample_bytree=0.705051979426657, gamma=3.6386287158866253, learning_rate=0.2805317196658718, max_depth=6, min_child_weight=7, n_estimators=390, subsample=0.675465667449572; total time= 7.8min
[CV] END colsample_bytree=0.9186941777766422, gamma=4.416576386904311, learning_rate=0.09798893186641074, max_depth=12, min_child_weight=9, n_estimators=144, subsample=0.5691542691392876; total time= 3.4min
[CV] END colsample_bytree=0.7224162561505759, gamma=4.593612608346885, learning_rate=0.04040915598028404, max_depth=7, min_child_weight=3, n_estimators=418, subsample=0.5777240270252717; total time= 8.6min
[CV] END colsample_bytree=0.7224162561505759, gamma=4.593612608346885, learning_rate=0.0

BayesSearchCV(cv=5,
              estimator=XGBClassifier(base_score=None, booster=None,
                                      callbacks=None, colsample_bylevel=None,
                                      colsample_bynode=None,
                                      colsample_bytree=None, device=None,
                                      early_stopping_rounds=None,
                                      enable_categorical=False,
                                      eval_metric='mlogloss',
                                      feature_types=None, feature_weights=None,
                                      gamma=None, grow_policy=None,
                                      importance_type=None,
                                      interaction_constrai...
                             'learning_rate': Real(low=0.01, high=0.3, prior='uniform', transform='normalize'),
                             'max_depth': Integer(low=3, high=12, prior='uniform', transform='normalize'),
                             'min_child_weight': Integer(low=1, high=10, prior='uniform', transform='normalize'),
                             'n_estimators': Integer(low=100, high=800, prior='uniform', transform='normalize'),
                             'subsample': Real(low=0.5, high=1.0, prior='uniform', transform='normalize')},
              verbose=2)

[CV] END colsample_bytree=0.705051979426657, gamma=3.6386287158866253, learning_rate=0.2805317196658718, max_depth=6, min_child_weight=7, n_estimators=390, subsample=0.675465667449572; total time= 7.8min
[CV] END colsample_bytree=0.9186941777766422, gamma=4.416576386904311, learning_rate=0.09798893186641074, max_depth=12, min_child_weight=9, n_estimators=144, subsample=0.5691542691392876; total time= 3.4min
[CV] END colsample_bytree=0.9186941777766422, gamma=4.416576386904311, learning_rate=0.09798893186641074, max_depth=12, min_child_weight=9, n_estimators=144, subsample=0.5691542691392876; total time= 1.8min
[CV] END colsample_bytree=0.7224162561505759, gamma=4.593612608346885, learning_rate=0.04040915598028404, max_depth=7, min_child_weight=3, n_estimators=418, subsample=0.5777240270252717; total time= 8.6min
[CV] END colsample_bytree=0.9061979941786817, gamma=0.8593578069828035, learning_rate=0.18343365247851712, max_depth=10, min_child_weight=6, n_estimators=167, subsample=0.87790

In [23]:
params = bayes_search.best_params_
params

OrderedDict([('colsample_bytree', 0.8997767208035865),
             ('gamma', 2.190145932204617),
             ('learning_rate', 0.16271986876703082),
             ('max_depth', 9),
             ('min_child_weight', 9),
             ('n_estimators', 602),
             ('subsample', 0.712089036230341)])

In [24]:
model = bayes_search.best_estimator_

In [26]:
y_pred = model.predict(x_test)

In [28]:
rev_mapping = {
    0: -1,
    1: 0,
    2: 1
}

y_pred = np.vectorize(rev_mapping.get)(y_pred)
y_test = np.vectorize(rev_mapping.get)(y_test)

In [32]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
classification_rep = classification_report(
    y_test,
    y_pred,
    output_dict=True
)
print(classification_rep)


{'-1': {'precision': 0.7822242206235012, 'recall': 0.6026558891454965, 'f1-score': 0.6807983302895904, 'support': 8660.0}, '0': {'precision': 0.7298860447803929, 'recall': 0.9352116809743929, 'f1-score': 0.8198893606072302, 'support': 13629.0}, '1': {'precision': 0.868015475359929, 'recall': 0.7770396865951286, 'f1-score': 0.8200119832234871, 'support': 17613.0}, 'accuracy': 0.7932183850433562, 'macro avg': {'precision': 0.7933752469212744, 'recall': 0.7716357522383394, 'f1-score': 0.7735665580401024, 'support': 39902.0}, 'weighted avg': {'precision': 0.8022163105214256, 'recall': 0.7932183850433562, 'f1-score': 0.789756320398453, 'support': 39902.0}}


In [33]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.7932


In [41]:

try:
    with mlflow.start_run() as run:


        mlflow.set_tag("model", "XGBoost")
        mlflow.set_tag("optimization", "Bayesian Search")


        mlflow.log_param("vectorizer_type", "CountVectorizer")
        mlflow.log_param("vectorizer_max_features", 1000)
        mlflow.log_params(params)


        mlflow.log_metric("accuracy", accuracy)

        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric_name, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric_name}", value)


        mlflow.xgboost.log_model(
            xgb_model=model,
            artifact_path="xgboost_model"
        )

        data.to_csv("dataset.csv", index=False)
        mlflow.log_artifact("dataset.csv")

        # Confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)

        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title("Confusion Matrix")

        plt.savefig("confusion_matrix.png")
        plt.close()

        mlflow.log_artifact("confusion_matrix.png")

        print(f"Run ID: {run.info.run_id}")

except Exception as e:
    print(f"MLflow logging failed: {e}")

finally:
    
    import os

    if os.path.exists("dataset.csv"):
        os.remove("dataset.csv")

    if os.path.exists("confusion_matrix.png"):
        os.remove("confusion_matrix.png")

2026/07/25 21:20:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run ID: 44f0333b40344d4aa05aacd5f306abcf
🏃 View run suave-squirrel-259 at: http://ec2-13-50-105-122.eu-north-1.compute.amazonaws.com:5000/#/experiments/1/runs/44f0333b40344d4aa05aacd5f306abcf
🧪 View experiment at: http://ec2-13-50-105-122.eu-north-1.compute.amazonaws.com:5000/#/experiments/1
